# Bölüm 2/7 — Ozellik Muhendisligi


## Önceki bölümden devam
Bu hücre, `1_...` bölümünde kaydedilen tüm değişkenleri ve tanımlı fonksiyonları geri yükler.

In [ ]:
!pip install dill --quiet
import dill
dill.load_session('checkpoint_1.pkl')
print('Önceki bölümün oturumu yüklendi.')

## Metin/hedef ayrımı ve eğitim-test bölmesi

Temizlenmiş metin (`clean_text`) ve hedef değişken (`is_disinformation`), sınıf oranı korunacak şekilde (`stratify`) %80 eğitim / %20 test olarak ikiye bölünür.

In [ ]:
X_text = df_clean["clean_text"]
X_news_type = df_clean[["news_type"]]   # 2D olmali, OneHotEncoder icin
y = df_clean["is_disinformation"]

X_train_text, X_test_text, X_train_news, X_test_news, y_train, y_test = train_test_split(
    X_text, X_news_type, y, test_size=0.20, random_state=42, stratify=y
)

print("Eğitim veri sayısı:", len(X_train_text))
print("Test veri sayısı:", len(X_test_text))
print("\nEğitim sınıf dağılımı:\n", y_train.value_counts())
print("\nTest sınıf dağılımı:\n", y_test.value_counts())

## 11. TF-IDF vektörleştirme

`TfidfVectorizer` ile metinler sayısal vektörlere dönüştürülür. `stop_words="english"` yaygın İngilizce dolgu kelimelerini, `min_df=3`/`max_df=0.9` nadir/aşırı yaygın kelimeleri filtreler, `ngram_range=(1,2)` hem tekil kelimeleri hem ikili kelime gruplarını (bigram) özellik olarak kullanır.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.9, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("TF-IDF eğitim veri boyutu:", X_train_tfidf.shape)
print("TF-IDF test veri boyutu:", X_test_tfidf.shape)

## TF-IDF kelime dağarcığı kontrolü

Oluşan TF-IDF kelime dağarcığının toplam boyutu ve ilk 20 özelliği (alfabetik sırayla) görüntülenerek vektörleştirmenin beklenen şekilde çalıştığı doğrulanır.

In [ ]:
feature_names = tfidf.get_feature_names_out()

print("Toplam özellik (kelime/kelime öbeği) sayısı:", len(feature_names))
print("\nİlk 20 özellik:")
print(feature_names[:20])

## 12. (Opsiyonel) CountVectorizer karşılaştırması

Karşılaştırma amacıyla, TF-IDF ile aynı parametrelere sahip bir `CountVectorizer` (ham kelime sayımı) oluşturulur.

In [ ]:
count_vectorizer = CountVectorizer(stop_words="english", min_df=3, max_df=0.9, ngram_range=(1, 2))

X_train_count = count_vectorizer.fit_transform(X_train_text)
X_test_count = count_vectorizer.transform(X_test_text)

print("CountVectorizer eğitim veri boyutu:", X_train_count.shape)
print("CountVectorizer test veri boyutu:", X_test_count.shape)

`CountVectorizer`'ın ürettiği kelime dağarcığının boyutu ve ilk 20 özelliği görüntülenir.

In [ ]:
count_features = count_vectorizer.get_feature_names_out()

print("CountVectorizer toplam özellik sayısı:", len(count_features))
print("\nİlk 20 özellik:")
print(count_features[:20])

TF-IDF ve `CountVectorizer`'ın aynı kelime dağarcığını üretip üretmediği (`np.array_equal`) karşılaştırılarak iki vektörleştiricinin tutarlılığı doğrulanır.

In [ ]:
# TF-IDF ve CountVectorizer'ın aynı kelime dağarcığını ürettiğini doğrulamak için karşılaştır
ayni_mi = np.array_equal(count_features, feature_names)
print("İki vektörleştiricinin kelime dağarcığı birebir aynı mı?", ayni_mi)

print("\nSon 20 CountVectorizer özelliği:")
print(count_features[-20:])
print("\nSon 20 TF-IDF özelliği:")
print(feature_names[-20:])

## Sayısal-sadece token'ların temizlenmesi


Yalnızca rakam ve boşluktan oluşan (örn. tarih kalıntısı "03 2023" gibi) anlamsız token'lar tespit edilip kelime dağarcığından çıkarılır. "000 soldiers" gibi sayı+kelime ikilileri bilinçli olarak korunur, çünkü dezenformasyon tespiti için potansiyel bir sinyal taşıyabilir.

In [ ]:
import re as _re

def sayisal_sadece_mi(token):
    return bool(_re.fullmatch(r"[\d\s]+", token))

sayisal_mask = np.array([sayisal_sadece_mi(f) for f in feature_names])
print("Sayisal-sadece token sayisi (TF-IDF/Count vocabulary ayni oldugu icin ikisi icin de gecerli):", sayisal_mask.sum())
print("Ornekler:", feature_names[sayisal_mask][:15])

keep_idx = np.where(~sayisal_mask)[0]

# TF-IDF matrislerini ve feature_names'i guncelle
X_train_tfidf = X_train_tfidf[:, keep_idx]
X_test_tfidf = X_test_tfidf[:, keep_idx]
feature_names = feature_names[keep_idx]

# CountVectorizer matrislerini de tutarlilik icin ayni sekilde guncelle
X_train_count = X_train_count[:, keep_idx]
X_test_count = X_test_count[:, keep_idx]
count_features = count_features[keep_idx]

print("\nTemizlik sonrasi TF-IDF egitim boyutu:", X_train_tfidf.shape)
print("Temizlik sonrasi TF-IDF test boyutu:", X_test_tfidf.shape)
print("\nIlk 20 ozellik (alfabetik, sayisal token'lar cikarilmis):")
print(feature_names[:20])


## `news_type` sütununun X'e eklenmesi

Şimdiye kadar model yalnızca metni (`clean_text` -> TF-IDF) girdi olarak kullanıyordu. Hocamızın isteği doğrultusunda, haber kategorisini (`news_type`) de bir özellik olarak modele eklemek için bu sütun `OneHotEncoder` ile sayısallaştırılıp TF-IDF matrisine yatay olarak (`hstack`) birleştirilir. **Sızıntı kontrolü:** encoder yalnızca eğitim verisiyle (`X_train_news`) `fit` edilir; test verisine sadece `transform` uygulanır — tıpkı yukarıdaki TF-IDF adımında olduğu gibi. `handle_unknown="ignore"` parametresi, test setinde eğitim setinde görülmemiş bir kategori çıkarsa (örn. çok nadir görülen `Health`/`Business` gibi kategoriler) hatanın önüne geçer.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

# news_type'i one-hot encode et (sadece train'e fit, test'e sadece transform)
ohe = OneHotEncoder(handle_unknown="ignore")
X_train_news_ohe = ohe.fit_transform(X_train_news)
X_test_news_ohe = ohe.transform(X_test_news)

print("news_type kategorileri:", ohe.categories_[0])
print("One-hot boyutu (train):", X_train_news_ohe.shape)

# TF-IDF ile news_type'i yan yana birlestir
X_train_tfidf = hstack([X_train_tfidf, X_train_news_ohe]).tocsr()
X_test_tfidf = hstack([X_test_tfidf, X_test_news_ohe]).tocsr()

print("\nBirlestirilmis X_train_tfidf boyutu:", X_train_tfidf.shape)
print("Birlestirilmis X_test_tfidf boyutu:", X_test_tfidf.shape)

# feature_names'i de guncelle: asagidaki katsayi/onem analizi hucreleri
# (Linear SVM, Random Forest, XGBoost, Multinomial NB) dogru isimlerle calissin diye
feature_names = np.concatenate([feature_names, ohe.get_feature_names_out()])
print("\nEklenen news_type ozellikleri:", list(ohe.get_feature_names_out()))

## Bu bölümü kaydet
Bir sonraki bölümün bu noktadan devam edebilmesi için tüm oturum (değişkenler, modeller, fonksiyonlar) diske kaydedilir.

In [ ]:
import dill
dill.dump_session('checkpoint_2.pkl')
print('Oturum checkpoint_2.pkl olarak kaydedildi.')